### The University of Melbourne, School of Computing and Information Systems
# COMP90086 Computer Vision, 2025 Semester 2

## FINAL PROJECT

In [27]:
import tensorflow as tf
import pandas as pd
import os
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

# --- Configuration ---
IMG_HEIGHT = 640
IMG_WIDTH = 480
BATCH_SIZE = 32
BASE_DIR = "Nutrition5k"
VALIDATION_SPLIT = 0.20 # 20% for validation
RANDOM_SEED = 42 # Use a seed for reproducible splits

# --- 1. Load the full labeled dataset manifest ---
full_labels_path = os.path.join(BASE_DIR, "nutrition5k_train.csv")
df = pd.read_csv(full_labels_path)
df['ID'] = df['ID'].astype(str)




In [37]:
# complete train and test split (with normalised pixel values)
df = df.assign(train_image_path=BASE_DIR + "\\train\\color\\" + df["ID"].astype(str) + "\\rgb.png",
               test_image_path=BASE_DIR + "\\test\\color\\" + df["ID"].astype(str) + "\\rgb.png")

datagen = ImageDataGenerator(rescale=1./255,
                                 validation_split=VALIDATION_SPLIT)

train_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='training' # For training data
    )


validation_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='validation' # For training data
    )

Found 2641 validated image filenames.
Found 660 validated image filenames.


In [43]:
os.listdir(os.path.join(BASE_DIR, "test/color"))

['dish_3301',
 'dish_3302',
 'dish_3303',
 'dish_3304',
 'dish_3305',
 'dish_3306',
 'dish_3307',
 'dish_3308',
 'dish_3309',
 'dish_3310',
 'dish_3311',
 'dish_3312',
 'dish_3313',
 'dish_3314',
 'dish_3315',
 'dish_3316',
 'dish_3317',
 'dish_3318',
 'dish_3319',
 'dish_3320',
 'dish_3321',
 'dish_3322',
 'dish_3323',
 'dish_3324',
 'dish_3325',
 'dish_3326',
 'dish_3327',
 'dish_3328',
 'dish_3329',
 'dish_3330',
 'dish_3331',
 'dish_3332',
 'dish_3333',
 'dish_3334',
 'dish_3335',
 'dish_3336',
 'dish_3337',
 'dish_3338',
 'dish_3339',
 'dish_3340',
 'dish_3341',
 'dish_3342',
 'dish_3343',
 'dish_3344',
 'dish_3345',
 'dish_3346',
 'dish_3347',
 'dish_3348',
 'dish_3349',
 'dish_3350',
 'dish_3351',
 'dish_3352',
 'dish_3353',
 'dish_3354',
 'dish_3355',
 'dish_3356',
 'dish_3357',
 'dish_3358',
 'dish_3359',
 'dish_3360',
 'dish_3361',
 'dish_3362',
 'dish_3363',
 'dish_3364',
 'dish_3365',
 'dish_3366',
 'dish_3367',
 'dish_3368',
 'dish_3369',
 'dish_3370',
 'dish_3371',
 'dish

In [48]:
# create dataset to load test dataset
df_test = pd.DataFrame({"ID": os.listdir(os.path.join(BASE_DIR, "test/color"))})
df_test = df_test.assign(test_image_path = BASE_DIR + "\\test\\color\\" + df_test["ID"].astype(str) + "\\rgb.png")

In [49]:
df_test

,ID,test_image_path
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png
...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png


In [50]:
test_datagen=ImageDataGenerator(rescale=1./255.)
test_generator=test_datagen.flow_from_dataframe(
dataframe=df_test,
directory=os.getcwd(),
x_col="test_image_path",
y_col=None,
target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
batch_size=BATCH_SIZE,
seed=RANDOM_SEED,
shuffle=False,
class_mode=None)

Found 189 validated image filenames.


In [ ]:
# build basic regression model
model = tf.keras.Sequential(
    [
        layers.Input((IMG_HEIGHT, IMG_WIDTH, 3)),
        
        layers.Conv2D(8, (5, 5), activation='relu'), # fill in
        layers.MaxPooling2D((2, 2)), # fill in
        
        layers.Flatten(),
        layers.Dense(10, activation="relu"),
        layers.Dense(1, activation="linear")
    ], 
)

model.compile(optimizer='adam', loss='mse', metrics=['mse'])

In [34]:
STEP_SIZE_TRAIN = train_generator.n//train_generator.batch_size
STEP_SIZE_VALID = validation_generator.n//validation_generator.batch_size

model.fit(train_generator,
          steps_per_epoch=STEP_SIZE_TRAIN,
          validation_data=validation_generator,
          validation_steps=STEP_SIZE_VALID,
          epochs=10
)


Epoch 1/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 55s 658ms/step - loss: 62158.8945 - mae: 162.7327 - val_loss: 26663.2910 - val_mae: 124.9135
Epoch 2/10
 1/82 ━━━━━━━━━━━━━━━━━━━━ 20s 251ms/step - loss: 14778.6562 - mae: 96.1326

c:\Users\nares\anaconda3\envs\CV\lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


82/82 ━━━━━━━━━━━━━━━━━━━━ 8s 97ms/step - loss: 14778.6562 - mae: 96.1326 - val_loss: 24567.8027 - val_mae: 117.7040
Epoch 3/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 53s 647ms/step - loss: 25921.1543 - mae: 113.0865 - val_loss: 31043.8496 - val_mae: 122.2036
Epoch 4/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 8s 97ms/step - loss: 25688.2344 - mae: 116.3270 - val_loss: 30381.0059 - val_mae: 121.4276
Epoch 5/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 50s 611ms/step - loss: 29387.5059 - mae: 114.0500 - val_loss: 21360.7461 - val_mae: 108.4167
Epoch 6/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 8s 98ms/step - loss: 10358.8672 - mae: 74.7965 - val_loss: 21802.1855 - val_mae: 109.5734
Epoch 7/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 50s 613ms/step - loss: 24669.8027 - mae: 105.5807 - val_loss: 20227.3145 - val_mae: 102.3912
Epoch 8/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 8s 92ms/step - loss: 20362.9102 - mae: 104.5949 - val_loss: 19587.8887 - val_mae: 103.1120
Epoch 9/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 47s 578ms/step - loss: 21380.7695 - mae: 97.6183 - val_loss: 17991

In [55]:
# validation
STEP_SIZE_TEST=test_generator.n//test_generator.batch_size
model.evaluate(generator=validation_generator,
steps=STEP_SIZE_TEST)

ValueError: Arguments not recognized: {'generator': <keras.src.legacy.preprocessing.image.DataFrameIterator object at 0x00000202AC489B50>}

In [59]:
test_generator.reset()
preds=model.predict(test_generator)

6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 354ms/step


In [60]:
df_test = df_test.assign(Value=preds)

In [61]:
df_test

,ID,test_image_path,Value
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png,592.266663
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png,381.901947
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png,205.510010
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png,306.300171
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png,454.631714
...,...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png,160.208313
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png,3.290866
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png,172.967957
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png,262.092773
